# 🔍 Diagnostic stock — réceptions fournisseur (Sochepress)

Ce programme est en **LECTURE SEULE** : il ne modifie rien dans Odoo.
Il collecte les réceptions récentes du fournisseur, les mouvements de stock
et l'état actuel des quantités, puis produit un rapport à télécharger.

## Mode d'emploi
1. Cliquez sur **▶** à gauche de la cellule ci-dessous
2. **Entrée** pour accepter l'URL, la base et le fournisseur proposés
3. Indiquez depuis combien de jours chercher (Entrée = 30 jours)
4. Entrez votre login et mot de passe Odoo
5. À la fin, le fichier **diagnostic_stock.txt** se télécharge : envoyez-le
   dans la conversation Claude pour analyse

In [ ]:
# -*- coding: utf-8 -*-
import getpass, sys, xmlrpc.client
from datetime import datetime, timedelta

ODOO_URL = "http://94.130.90.253:9069"
ODOO_DB = "agora-prod"

url = input(f"URL Odoo [{ODOO_URL}] : ").strip() or ODOO_URL
db = input(f"Base de données [{ODOO_DB}] : ").strip() or ODOO_DB
fournisseur = input("Nom du fournisseur [sochepress] : ").strip() or "sochepress"
jours = int(input("Chercher sur combien de jours en arrière ? [30] : ").strip() or "30")
login = input("Login Odoo : ").strip()
mdp = getpass.getpass("Mot de passe Odoo : ")

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", allow_none=True)
uid = common.authenticate(db, login, mdp, {})
if not uid:
    sys.exit("Erreur : authentification refusée")
modeles = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)
print(f"✔ Connecté (lecture seule) — utilisateur #{uid}")

def lire(modele, domaine, champs, limite=300, tri=None):
    kw = {"fields": champs, "limit": limite}
    if tri: kw["order"] = tri
    return modeles.execute_kw(db, uid, mdp, modele, "search_read", [domaine], kw)

def rel(v):
    return v[1] if isinstance(v, (list, tuple)) and len(v) > 1 else (v or "")

depuis = (datetime.now() - timedelta(days=jours)).strftime("%Y-%m-%d")
R = [f"DIAGNOSTIC STOCK — fournisseur ~ '{fournisseur}' — depuis {depuis} — généré {datetime.now():%Y-%m-%d %H:%M}"]

# 1. Fournisseur(s)
partenaires = lire("res.partner", [["name", "ilike", fournisseur]], ["name"])
pids = [p["id"] for p in partenaires]
R.append(f"\n═══ 1. FOURNISSEURS TROUVÉS : {len(partenaires)} ═══")
for p in partenaires:
    R.append(f"  [{p['id']}] {p['name']}")
if not pids:
    R.append("  ⚠ Aucun partenaire trouvé — vérifiez l'orthographe")

# 2. Transferts (réceptions/retours) du fournisseur
pickings = lire("stock.picking",
    ["|", ["partner_id", "in", pids], ["partner_id.parent_id", "in", pids]],
    ["name", "state", "picking_type_id", "origin", "partner_id",
     "scheduled_date", "date_done", "create_date", "backorder_id", "write_date", "write_uid"],
    limite=100, tri="create_date desc")
pickings = [p for p in pickings if (p.get("create_date") or "") >= depuis or (p.get("date_done") or "") >= depuis or (p.get("write_date") or "") >= depuis]
R.append(f"\n═══ 2. TRANSFERTS RÉCENTS DU FOURNISSEUR : {len(pickings)} ═══")
for p in pickings:
    R.append(f"  {p['name']} | type: {rel(p['picking_type_id'])} | état: {p['state']}"
             f" | origine: {p.get('origin') or '-'} | fait le: {p.get('date_done') or '-'}"
             f" | créé: {p['create_date']} | modifié: {p['write_date']} par {rel(p.get('write_uid'))}"
             + (f" | RELIQUAT de {rel(p['backorder_id'])}" if p.get("backorder_id") else ""))

# 3. Mouvements de ces transferts
pk_ids = [p["id"] for p in pickings]
moves = lire("stock.move", [["picking_id", "in", pk_ids]],
    ["picking_id", "product_id", "product_uom_qty", "quantity", "state",
     "location_id", "location_dest_id", "date"], limite=1000) if pk_ids else []
R.append(f"\n═══ 3. LIGNES DE CES TRANSFERTS : {len(moves)} ═══")
R.append("  (demandé vs fait — un écart ou un doublon ici est souvent la cause)")
for m in moves:
    R.append(f"  {rel(m['picking_id'])} | {rel(m['product_id'])[:55]}"
             f" | demandé: {m['product_uom_qty']} | fait: {m.get('quantity')}"
             f" | état: {m['state']} | {rel(m['location_id'])} → {rel(m['location_dest_id'])} | {m['date']}")

# 4. Produits concernés : stock actuel
prod_ids = sorted({m["product_id"][0] for m in moves if m.get("product_id")})
produits = lire("product.product", [["id", "in", prod_ids]],
    ["name", "barcode", "qty_available", "virtual_available"], limite=1000) if prod_ids else []
R.append(f"\n═══ 4. STOCK ACTUEL DES PRODUITS CONCERNÉS : {len(produits)} ═══")
for p in produits:
    R.append(f"  [{p['id']}] {p.get('barcode') or '-'} | {p['name'][:55]}"
             f" | en stock: {p['qty_available']} | prévu: {p['virtual_available']}")

# 5. TOUS les mouvements faits récents sur ces produits (pour repérer doublons/ajustements)
tous_mv = lire("stock.move",
    [["product_id", "in", prod_ids], ["date", ">=", depuis], ["state", "=", "done"]],
    ["reference", "product_id", "quantity", "is_inventory",
     "location_id", "location_dest_id", "date", "picking_id"],
    limite=2000, tri="date asc") if prod_ids else []
R.append(f"\n═══ 5. TOUS LES MOUVEMENTS FAITS SUR CES PRODUITS DEPUIS {depuis} : {len(tous_mv)} ═══")
for m in tous_mv:
    tag = " ⚠ AJUSTEMENT INVENTAIRE" if m.get("is_inventory") else ""
    R.append(f"  {m['date']} | {m.get('reference') or rel(m.get('picking_id'))}"
             f" | {rel(m['product_id'])[:45]} | qté: {m.get('quantity')}"
             f" | {rel(m['location_id'])} → {rel(m['location_dest_id'])}{tag}")

# 6. Quants par emplacement
quants = lire("stock.quant", [["product_id", "in", prod_ids]],
    ["product_id", "location_id", "quantity", "reserved_quantity"], limite=2000) if prod_ids else []
R.append(f"\n═══ 6. DÉTAIL PAR EMPLACEMENT (quants) : {len(quants)} ═══")
for q in quants:
    R.append(f"  {rel(q['product_id'])[:45]} | {rel(q['location_id'])}"
             f" | qté: {q['quantity']} | réservé: {q['reserved_quantity']}")

rapport = "\n".join(R)
print(rapport[:3000] + ("\n… (rapport complet dans le fichier)" if len(rapport) > 3000 else ""))
with open("diagnostic_stock.txt", "w", encoding="utf-8") as f:
    f.write(rapport)
try:
    from google.colab import files
    files.download("diagnostic_stock.txt")
except ImportError:
    print("Rapport enregistré : diagnostic_stock.txt")
print("\n✔ TERMINÉ — envoyez le fichier diagnostic_stock.txt dans la conversation Claude")